# 00_preprocessing — Build the modeling dataset (TOA → \(a_\phi\))

**Purpose.** Construct the **canonical dataset** for modeling. Load NOMAD–SeaWiFS matchups where **TOA radiances are already Rayleigh- and Fresnel-corrected in the raw file**, compute **base-10 logs** for predictors and responses, enforce **complete-case inclusion**, and export a tidy file for the modeling notebook.

**Scope.** Predictive pipeline only — *not* an IOP inversion. Predictors are **log\(_{10}\)** of corrected TOA radiances at SeaWiFS bands (412, 443, 490, 510, 555, 670 nm). Responses are **log\(_{10}\)** phytoplankton absorption \(a_\phi\) at **443, 555, 670 nm**. **No standardization** of predictors.

---

## Inputs
- Raw NOMAD/SeaWiFS matchup file(s) with **pre-corrected TOA radiances**.
- SeaWiFS band definitions (wavelength map).

## Outputs
- `data/processed/aph_toa_ready.parquet` (analysis-ready table).
- `data/processed/schema.json` (columns, dtypes, units, band lists).
- `figs/eda/pairplot_log_toa_aph.png` (KDE diagonals + pairwise scatter).
- `results/qa/qc_summary.csv` (row counts, missingness, inclusion stats).

---

## Steps
1. **Load & harmonize** raw data (confirm corrections already applied; no re-correction).
2. **Log transforms**: compute **log\(_{10}\)** for radiances and \(a_\phi\).
3. **Selection**: require **all six** predictor bands and **all three** response bands.
4. **QA/QC**: duplicates, missingness, basic stats.
5. **EDA (first section)**: **seaborn pairplot** on **log-domain** variables (KDE diagonals).
6. **Export**: dataset + schema + EDA figures.

---

## Reproducibility
Deterministic transforms; fixed filenames; schema JSON documents columns and units.

---

## First section: quick data survey
Load raw → construct log variables → **pairplot** (KDE diagonals) to inspect distributions and pairwise structure before modeling.


In [19]:
from pathlib import Path

import numpy as np
import pandas as pd
from scripts import data_loader

#### Data Card
Filename: Rayleigh&Fresnel_corrected_Rrc.csv

The parameters of interest are $Rrc$, **top of atmosphere Rayleigh and Fresnel-corrected radiance* and the associated geophysical parameters, Chl and phytoplankton absorption, aph, at various wavelengths.

Phytoplankton absorption, aph, is calculated from:

$$aph = ap - ad$$

In [20]:
project_path = Path.cwd()
data_path = project_path / 'data' 
fp = '01_raw/nomad_seawifs_Rayleigh&Fresnel_corrected.csv'

In [50]:
df = data_loader(fp=data_path / fp)
df_out = df.filter(regex='(^ad|^ap[0-9]+$)')
df_out.head()

,ap405,ap411,ap443,ap455,ap465,ap489,ap510,ap520,ap530,ap550,...,ad555,ad560,ad565,ad570,ad590,ad619,ad625,ad665,ad670,ad683
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.02880,0.03093,0.03554,0.03254,0.03137,0.02341,0.01485,0.01196,0.00955,0.00591,...,0.00136,0.00128,0.00121,0.00114,0.00091,0.00066,0.00062,0.00039,0.00037,0.00032
3,0.02313,0.02458,0.02654,0.02387,0.02274,0.01664,0.01087,0.00887,0.00714,0.00447,...,0.00082,0.00076,0.00072,0.00066,0.00051,0.00035,0.00033,0.00019,0.00017,0.00015
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_toa = df.filter(regex='^Rrc')
df_toa.info()

In [51]:
# PREPROCESS IOP
desired_lambda = [443, 555, 670]
cols = [f'{c}{i}' for c in ['ad', 'ap'] for i in desired_lambda]
df_ad_ap = df_out[cols]
df_aph = pd.DataFrame(columns = [f'aph{λ}' for λ in desired_lambda])
for λ in desired_lambda:
    df_aph[f'aph{λ}'] = df_ad_ap[f'ap{λ}'] - df_ad_ap[f'ad{λ}']
df_aph

,aph443,aph555,aph670
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,0.03078,0.00385,0.01040
3,0.02283,0.00310,0.00935
4,NaN,NaN,NaN
...,...,...,...
491,NaN,NaN,NaN
492,0.14267,0.02354,0.04603
493,NaN,NaN,NaN
494,0.04911,0.00697,0.01400


In [37]:
df_aph.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 496 entries, 0 to 495
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   aph443  168 non-null    float64
 1   aph555  166 non-null    float64
 2   aph670  168 non-null    float64
dtypes: float64(3)
memory usage: 11.8 KB


In [52]:
# SAVE INTERIM DATA STATE
df_aph.to_parquet(data_path / '02_reduced_columns' / 'df_aph.pqt')
df_toa.to_parquet(data_path / '02_reduced_columns' / 'df_toa.pqt')

In [53]:
# LOG TRANSFORM DATA
df_all = pd.concat((df_toa, df_aph), axis=1)
df_log = df_all.replace(0, np.nan).transform(np.log10)
column_name_mapper = {k: f'log_{k}' for k in df_all.columns} 
df_log.rename(columns=column_name_mapper, inplace=True)
df_log

,log_Rrc_412,log_Rrc_443,log_Rrc_490,log_Rrc_510,log_Rrc_555,log_Rrc_670,log_aph443,log_aph555,log_aph670
0,-1.917660,-1.905994,-1.930380,-1.975543,-2.040501,-2.176858,NaN,NaN,NaN
1,-1.977774,-1.973214,-2.017116,-2.101675,-2.205903,-2.319302,NaN,NaN,NaN
2,-2.352334,-2.357865,-2.372588,-2.433492,-2.577377,-2.929113,-1.511731,-2.414539,-1.982967
3,-2.231428,-2.231674,-2.256908,-2.333192,-2.478052,-2.757732,-1.641494,-2.508638,-2.029188
4,-2.023925,-2.047304,-2.112433,-2.223764,-2.380771,-2.599169,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
491,-2.345332,-2.275178,-2.171985,-2.161233,-2.166044,-2.577262,NaN,NaN,NaN
492,-2.180295,-2.167633,-2.165241,-2.155715,-2.111349,-2.305298,-0.845667,-1.628194,-1.336959
493,-1.983673,-1.916870,-1.831338,-1.870797,-1.954302,-2.383724,NaN,NaN,NaN
494,-2.295468,-2.234034,-2.153897,-2.141541,-2.146016,-2.473777,-1.308830,-2.156767,-1.853872


In [ ]:
# SAVE FINAL STATE - Data is ready for study
final_path = data_path / '03_transformed'
df_log.to_parquet(final_path / 'df_log.pqt')
df_log.to_csv(final_path / 'df_log.csv')